# BBO Function 1 — Part 2 reflection analysis

This notebook is for the **first 2D unknown function**.

It is designed to help answer the Part 2 reflection prompts:
- which evaluated inputs behaved like support vectors or boundary points;
- how surrogate gradients change with the inputs;
- how classification framing separates “good” and “bad” outputs;
- whether linear regression, SVM, or neural networks are most useful;
- which variables influence the surrogate most;
- whether the neural network captures nonlinear patterns better than simpler models.

Assumption: this BBO objective is being **minimised**.


## Version 3: neural network surrogate + input gradients

Use this version for the backpropagation and variable influence questions.

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load original function 1 data.
# Run this notebook from the same folder where initial_data/function_1 exists.
input_data = np.load('../initial_data/function_1/initial_inputs.npy')
output_data = np.load('../initial_data/function_1/initial_outputs.npy')

# Add your latest evaluated point here.
# You said the new input is (0.5, 0.5). Replace NEW_OUTPUT with the portal output.
NEW_POINT = np.array([[0.5, 0.5]])
NEW_OUTPUT = 2.6752879910742468e-9

if NEW_OUTPUT is not None:
    input_data = np.vstack([input_data, NEW_POINT])
    output_data = np.append(output_data, NEW_OUTPUT)

df = pd.DataFrame(input_data, columns=['x1', 'x2'])
df['y'] = output_data
df['log_abs_y'] = np.log(np.abs(output_data) + 1e-300)
df['rank_min'] = df['y'].rank(method='min', ascending=True).astype(int)

print(df.sort_values('y').to_string(index=True))
print("\nBest point so far:")
print(df.loc[df['y'].idxmin()])


          x1        x2              y   log_abs_y  rank_min
4   0.650114  0.681526  -3.606063e-03   -5.625139         1
5   0.410437  0.147554  -2.159249e-54 -123.569835         2
6   0.312691  0.078723  -2.089093e-91 -208.798513         3
3   0.840353  0.264732  3.341771e-124 -284.314051         4
8   0.082507  0.403488   3.606771e-81 -185.226580         5
0   0.319404  0.762959   1.322677e-79 -181.624565         6
9   0.883890  0.582254   6.229856e-48 -108.694731         7
1   0.574329  0.879898   1.033078e-46 -105.886371         8
7   0.683418  0.861057   2.535001e-40  -91.173210         9
2   0.731024  0.733000   7.710875e-16  -34.798730        10
10  0.500000  0.500000   2.675288e-09  -19.739209        11

Best point so far:
x1           0.650114
x2           0.681526
y           -0.003606
log_abs_y   -5.625139
rank_min     1.000000
Name: 4, dtype: float64


In [ ]:

plt.figure(figsize=(7, 6))
sc = plt.scatter(df['x1'], df['x2'], c=df['log_abs_y'], s=100, edgecolors='black')
plt.colorbar(sc, label='log(|y|)')
for i, row in df.iterrows():
    plt.annotate(str(i), (row['x1'], row['x2']), xytext=(6, 6), textcoords='offset points')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Observed function values for Function 1')
plt.grid(alpha=0.25)
plt.show()


In [ ]:

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

X_np = input_data.astype(np.float32)
# Use log output for numerical stability if outputs vary over many orders of magnitude.
y_np = np.log(np.abs(output_data) + 1e-300).astype(np.float32).reshape(-1, 1)

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_np).astype(np.float32)
y_scaled = y_scaler.fit_transform(y_np).astype(np.float32)

X_t = torch.tensor(X_scaled)
y_t = torch.tensor(y_scaled)

class SmallSurrogateNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)

model = SmallSurrogateNN()
opt = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(3000):
    opt.zero_grad()
    pred = model(X_t)
    loss = loss_fn(pred, y_t)
    loss.backward()
    opt.step()

    if epoch % 500 == 0:
        print(epoch, float(loss))

with torch.no_grad():
    fitted_scaled = model(X_t).numpy()
    fitted_log_y = y_scaler.inverse_transform(fitted_scaled).ravel()

df_nn = df.copy()
df_nn['nn_pred_log_abs_y'] = fitted_log_y
df_nn['nn_residual_log'] = df_nn['log_abs_y'] - df_nn['nn_pred_log_abs_y']
print(df_nn[['x1','x2','y','log_abs_y','nn_pred_log_abs_y','nn_residual_log']].to_string(index=True))


In [ ]:

# Compute gradients dy_hat/dx at observed points.
# Because the model is trained on scaled x and scaled log-y, gradients are most useful comparatively.
X_grad = torch.tensor(X_scaled, requires_grad=True)
pred = model(X_grad)
grads = []
for i in range(len(X_grad)):
    model.zero_grad()
    if X_grad.grad is not None:
        X_grad.grad.zero_()
    pred[i].backward(retain_graph=True)
    grads.append(X_grad.grad[i].detach().numpy().copy())

grads = np.array(grads)
df_grad = df.copy()
df_grad['grad_x1_scaled'] = grads[:, 0]
df_grad['grad_x2_scaled'] = grads[:, 1]
df_grad['grad_norm'] = np.linalg.norm(grads, axis=1)
df_grad['dominant_variable'] = np.where(np.abs(grads[:,0]) >= np.abs(grads[:,1]), 'x1', 'x2')

print(df_grad.sort_values('grad_norm', ascending=False)[['x1','x2','y','log_abs_y','grad_x1_scaled','grad_x2_scaled','grad_norm','dominant_variable']].to_string(index=True))

print("\\nAverage absolute gradient:")
print("x1:", np.mean(np.abs(grads[:,0])))
print("x2:", np.mean(np.abs(grads[:,1])))


In [ ]:

# NN surface and gradient field.
grid_n = 80
xx, yy = np.meshgrid(np.linspace(0, 1, grid_n), np.linspace(0, 1, grid_n))
grid = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)
grid_scaled = x_scaler.transform(grid).astype(np.float32)

grid_t = torch.tensor(grid_scaled, requires_grad=True)
pred_scaled = model(grid_t)
grad_grid = torch.autograd.grad(pred_scaled.sum(), grid_t)[0].detach().numpy()
pred_log = y_scaler.inverse_transform(pred_scaled.detach().numpy()).reshape(grid_n, grid_n)

plt.figure(figsize=(7, 6))
cs = plt.contourf(xx, yy, pred_log, levels=30)
plt.colorbar(cs, label='NN predicted log(|y|)')
plt.scatter(input_data[:,0], input_data[:,1], c='white', edgecolors='black', s=70)
step = 6
plt.quiver(xx[::step, ::step], yy[::step, ::step],
           -grad_grid[:,0].reshape(grid_n, grid_n)[::step, ::step],
           -grad_grid[:,1].reshape(grid_n, grid_n)[::step, ::step],
           angles='xy')
plt.xlabel('x1'); plt.ylabel('x2')
plt.title('NN surrogate surface with negative-gradient directions')
plt.show()


In [ ]:

# Candidate selection by following low predicted output and strong negative-gradient directions.
rng = np.random.default_rng(42)
candidates = rng.random((10000, 2)).astype(np.float32)
cand_scaled = x_scaler.transform(candidates).astype(np.float32)

cand_t = torch.tensor(cand_scaled)
with torch.no_grad():
    cand_pred_log = y_scaler.inverse_transform(model(cand_t).numpy()).ravel()

best_idx = np.argmin(cand_pred_log)
best_next = candidates[best_idx]

print(f"NN suggested next query: [{best_next[0]:.6f}, {best_next[1]:.6f}]")
print(f"Predicted log(|y|): {cand_pred_log[best_idx]:.6f}")


## Reflection evidence from this version

Use this notebook to say:

The neural network surrogate allowed me to use backpropagation to compute local input gradients. The larger the absolute gradient for a variable, the more sensitive the predicted output is to that variable near the observed points. The negative-gradient arrows suggest directions in input space where the surrogate expects the function value to decrease. Because the data set is small, I would treat these directions as qualitative guidance rather than as exact optimisation steps.